In [ ]:

import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import os
import random
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score
import time

# Enable mixed precision training
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("Mixed precision enabled")

# Verify TensorFlow and GPU setup
print()
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU is {'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE'}")
print()

# Enable GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPU")
        tf.config.set_visible_devices(gpus[0], 'GPU')
        print("Using GPU:", gpus[0])
    except RuntimeError as e:
        print("Error configuring GPU:", e)

# MobileNetV2 model
def mobilenetv2():
    base_model = keras.applications.MobileNetV2(
        include_top=False, weights='imagenet', input_shape=(128, 128, 3), pooling='avg'
    )
    for layer in base_model.layers[:-10]:
        layer.trainable = False
    outputs = keras.layers.Dense(128, dtype='float32')(base_model.output)
    return keras.Model(inputs=base_model.input, outputs=outputs)

# TripletFace with robust batch handling
class TripletFace(keras.utils.Sequence):
    def __init__(self, image_dir, batch_size=4, image_size=(128, 128), seed=42, max_persons=20, embedding_model=None, max_triplets=10000):
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.image_size = image_size
        self.embedding_model = embedding_model
        self.max_triplets = max_triplets
        all_persons = os.listdir(image_dir)
        self.random = random.Random(seed)
        if max_persons:
            all_persons = self.random.sample(all_persons, min(max_persons, len(all_persons)))
        self.imgs_path = {
            person: os.listdir(os.path.join(image_dir, person)) for person in all_persons
        }
        print(f"Initialized TripletFace with {len(self.imgs_path)} persons, {sum(len(imgs) for imgs in self.imgs_path.values())} images")
        self.embedding_cache = self.__load_or_compute_embeddings()
        self.__produce_triplet_batches()
        print(f"Generated {len(self.triplets)} triplets")

    def __preprocess_image(self, image_path):
        try:
            image = tf.io.read_file(image_path)
            image = tf.image.decode_jpeg(image, channels=3)
            image = tf.image.resize(image, self.image_size)
            image = keras.applications.mobilenet_v2.preprocess_input(image)
            return image
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return tf.zeros(self.image_size + (3,), dtype=tf.float32)

    def __compute_embedding(self, image_path):
        image = self.__preprocess_image(image_path)
        if tf.reduce_sum(image) == 0:  # Check for zero tensor (failed load)
            return np.zeros(128, dtype=np.float32)
        image = tf.expand_dims(image, axis=0)
        return self.embedding_model(image, training=False).numpy()[0]

    def __load_or_compute_embeddings(self):
        cache_file = os.path.join(self.image_dir, "embeddings.npy")
        try:
            if os.path.exists(cache_file):
                print(f"Loading embeddings from {cache_file}")
                embeddings = np.load(cache_file, allow_pickle=True).item()
                if all(person in embeddings for person in self.imgs_path):
                    print("All persons found in embedding cache")
                    return embeddings
                else:
                    print("Missing persons in embedding cache, recomputing...")
            else:
                print("No embedding cache found, computing...")
        except Exception as e:
            print(f"Error loading embeddings: {e}, recomputing...")
        
        if not self.embedding_model:
            return {}
        print("Computing embeddings...")
        embeddings = {}
        for person in self.imgs_path:
            embeddings[person] = [
                (path, self.__compute_embedding(os.path.join(self.image_dir, person, path)))
                for path in self.imgs_path[person]
            ]
        try:
            np.save(cache_file, embeddings)
            print(f"Saved embeddings to {cache_file}")
        except Exception as e:
            print(f"Error saving embeddings: {e}")
        return embeddings

    def __produce_triplet_batches(self):
        self.triplets = []
        temp = []
        for person in self.imgs_path:
            if len(self.imgs_path[person]) < 2:  # Skip if not enough images
                continue
            anchor_paths = self.random.sample(self.imgs_path[person], len(self.imgs_path[person]) // 2)
            positive_paths = list({*self.imgs_path[person]} - {*anchor_paths})
            
            for i, anchor in enumerate(anchor_paths):
                if i < len(positive_paths):
                    positive = positive_paths[i]
                else:
                    positive = self.random.choice(positive_paths)
                    
                anchor_img = os.path.join(self.image_dir, person, anchor)
                positive_img = os.path.join(self.image_dir, person, positive)
                negative_person = self.random.choice(list({*self.imgs_path.keys()} - {person}))
                
                if self.embedding_model and self.embedding_cache:
                    try:
                        anchor_emb = next(e[1] for e in self.embedding_cache[person] if e[0] == anchor)
                        negative_candidates = self.embedding_cache[negative_person]
                        distances = [
                            np.sum(np.square(anchor_emb - emb[1])) for emb in negative_candidates
                        ]
                        negative_idx = np.argmin(distances)
                        negative_img = os.path.join(self.image_dir, negative_person, negative_candidates[negative_idx][0])
                    except (KeyError, StopIteration):
                        negative_img = os.path.join(
                            self.image_dir, negative_person, self.random.choice(self.imgs_path[negative_person])
                        )
                else:
                    negative_img = os.path.join(
                        self.image_dir, negative_person, self.random.choice(self.imgs_path[negative_person])
                    )
                temp.append((anchor_img, positive_img, negative_img))
                if len(temp) >= self.max_triplets:
                    break
            if len(temp) >= self.max_triplets:
                break
        self.random.shuffle(temp)
        num_batches = len(temp) // self.batch_size
        self.triplets = [temp[i * self.batch_size:(i + 1) * self.batch_size] for i in range(num_batches)]

    def on_epoch_end(self):
        self.__produce_triplet_batches()

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        try:
            batch = self.triplets[idx]
            anchor_paths, positive_paths, negative_paths = zip(*batch)
            anchor_batch = tf.stack([self.__preprocess_image(path) for path in anchor_paths])
            positive_batch = tf.stack([self.__preprocess_image(path) for path in positive_paths])
            negative_batch = tf.stack([self.__preprocess_image(path) for path in negative_paths])
            return anchor_batch, positive_batch, negative_batch
        except Exception as e:
            print(f"Error in __getitem__ at index {idx}: {e}")
            return (tf.zeros((self.batch_size, *self.image_size, 3), dtype=tf.float32),
                    tf.zeros((self.batch_size, *self.image_size, 3), dtype=tf.float32),
                    tf.zeros((self.batch_size, *self.image_size, 3), dtype=tf.float32))

# Convert TripletFace to tf.data.Dataset with finite cardinality
def triplet_face_to_dataset(triplet_face):
    def generator():
        for idx in range(len(triplet_face)):
            yield triplet_face[idx]
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32)
        )
    )
    dataset = dataset.cache(filename=f'triplet_cache_{random.randint(0, 1000000)}')  # Unique cache file
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    dataset = dataset.take(len(triplet_face))  # Limit to expected number of batches
    return dataset

# Custom metric to track triplet loss (fixed method name)
class TripletLossMetric(keras.metrics.Metric):
    def __init__(self, name='triplet_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_sum = self.add_weight(name='loss_sum', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, loss, sample_weight=None):
        self.loss_sum.assign_add(tf.reduce_sum(loss))
        self.count.assign_add(tf.cast(tf.size(loss), tf.float32))

    def result(self):
        return self.loss_sum / tf.maximum(self.count, 1.0)

    def reset_state(self):  # Renamed from reset_states
        self.loss_sum.assign(0.0)
        self.count.assign(0.0)

# Fixed SiameseModel with proper training loop
class SiameseModel(keras.Model):
    def __init__(self, siamese, loss="euclidean", margin=1.0):
        super().__init__()
        self.siamese = siamese
        self.loss_type = loss
        self.margin = margin
        self.triplet_loss_metric = TripletLossMetric()

    def __distance(self, x1, x2):
        if self.loss_type == "euclidean":
            return tf.reduce_sum(tf.square(x1 - x2), axis=-1)
        else:
            x1_norm = tf.linalg.normalize(x1, axis=-1)[0]
            x2_norm = tf.linalg.normalize(x2, axis=-1)[0]
            return 1 - tf.reduce_sum(x1_norm * x2_norm, axis=-1)

    def __triplet_loss(self, ap_distance, an_distance):
        loss = tf.maximum(ap_distance - an_distance + self.margin, 0)
        return tf.clip_by_value(loss, 0.0, 100.0)

    def call(self, inputs, training=False):
        anchor, positive, negative = inputs
        # Get embeddings
        anchor_embedding = self.siamese(anchor, training=training)
        positive_embedding = self.siamese(positive, training=training)
        negative_embedding = self.siamese(negative, training=training)
        
        # Calculate distances
        ap_distance = self.__distance(anchor_embedding, positive_embedding)
        an_distance = self.__distance(anchor_embedding, negative_embedding)
        loss = self.__triplet_loss(ap_distance, an_distance)
        
        return tf.reduce_mean(loss)

    def train_step(self, data):
        # Unpack data
        anchor, positive, negative = data
        
        with tf.GradientTape() as tape:
            # Forward pass
            loss = self.call((anchor, positive, negative), training=True)
            
            # Handle mixed precision
            if hasattr(self.optimizer, 'get_scaled_loss'):
                scaled_loss = self.optimizer.get_scaled_loss(loss)
            else:
                scaled_loss = loss
        
        # Compute gradients
        trainable_vars = self.siamese.trainable_variables
        if hasattr(self.optimizer, 'get_scaled_loss'):
            scaled_gradients = tape.gradient(scaled_loss, trainable_vars)
            gradients = self.optimizer.get_unscaled_gradients(scaled_gradients)
        else:
            gradients = tape.gradient(scaled_loss, trainable_vars)
        
        # Apply gradients
        if gradients and all(g is not None for g in gradients):
            # Clip gradients
            gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
            self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        
        # Update metrics
        self.triplet_loss_metric.update_state(loss)
        
        # Return metrics dictionary - this is crucial!
        return {
            "loss": loss,
            "triplet_loss": self.triplet_loss_metric.result()
        }

    def test_step(self, data):
        # Unpack data
        anchor, positive, negative = data
        
        # Forward pass
        loss = self.call((anchor, positive, negative), training=False)
        
        # Update metrics
        self.triplet_loss_metric.update_state(loss)
        
        # Return metrics dictionary
        return {
            "loss": loss,
            "triplet_loss": self.triplet_loss_metric.result()
        }

    def compute_loss(self, x=None, y=None, y_pred=None, sample_weight=None):
        # Override compute_loss to work with the custom loss
        return y_pred if y_pred is not None else 0.0

# Distance head
def distance_head(embedding, loss="euclidean"):
    assert loss == "euclidean" or loss == "cosine", "loss must be either 'euclidean' or 'cosine'"
    input_x1 = keras.Input(name="input_x1", shape=(128, 128, 3))
    input_x2 = keras.Input(name="input_x2", shape=(128, 128, 3))
    x1_embedding = embedding(input_x1)
    x2_embedding = embedding(input_x2)
    if loss == "euclidean":
        distance = tf.reduce_sum(tf.square(x1_embedding - x2_embedding), axis=-1)
    else:
        x1_norm = tf.linalg.normalize(x1_embedding, axis=-1)[0]
        x2_norm = tf.linalg.normalize(x2_embedding, axis=-1)[0]
        distance = 1 - tf.reduce_sum(x1_norm * x2_norm, axis=-1)
    return keras.Model(inputs=[input_x1, input_x2], outputs=distance)

# Classification head
def classification_head(embedding):
    inputs = keras.Input(shape=(128, 128, 3))
    x = embedding(inputs)
    x = keras.layers.Dense(512, dtype='float32')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)
    outputs = keras.layers.Dense(4000, activation="softmax", dtype='float32')(x)
    return keras.Model(inputs=inputs, outputs=outputs)

# Anti-Spoofing Module
def anti_spoofing_model():
    inputs = keras.Input(shape=(128, 128, 3))
    x = keras.layers.Conv2D(32, 3, activation='relu')(inputs)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.Conv2D(64, 3, activation='relu')(x)
    x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation='relu', dtype='float32')(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    return keras.Model(inputs=inputs, outputs=outputs)

# FaceVerificationSet
class FaceVerificationSet(keras.utils.Sequence):
    def __init__(self, base_data_dir, pairs_text_path, batch_size=4, image_size=(128, 128), seed=42):
        self.base_data_dir = base_data_dir
        self.pairs_text_path = pairs_text_path
        self.batch_size = batch_size
        self.image_size = image_size
        self.batches = []
        try:
            with open(self.pairs_text_path, encoding="utf-8") as file:
                self.data = [
                    {
                        "x1": os.path.join(base_data_dir, line.split(" ")[0]),
                        "x2": os.path.join(base_data_dir, line.split(" ")[1]),
                        "y": line.split(" ")[2].strip()
                    }
                    for line in file.readlines() if len(line.split(" ")) == 3
                ]
            num_batches = len(self.data) // self.batch_size
            self.batches = [self.data[i * self.batch_size:(i + 1) * self.batch_size] for i in range(num_batches)]
            print(f"Initialized FaceVerificationSet with {len(self.data)} pairs, {len(self.batches)} batches")
        except Exception as e:
            print(f"Error loading verification pairs: {e}")
            self.data = []
            self.batches = []

    def get_y(self):
        return [1 - int(data["y"]) for data in self.data]

    def __preprocess_image(self, image_path):
        try:
            image = tf.io.read_file(image_path)
            image = tf.image.decode_jpeg(image, channels=3)
            image = tf.image.resize(image, self.image_size)
            image = keras.applications.mobilenet_v2.preprocess_input(image)
            return image
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return tf.zeros(self.image_size + (3,), dtype=tf.float32)

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        try:
            batch = self.batches[idx]
            x1_batch = tf.stack([self.__preprocess_image(data["x1"]) for data in batch])
            x2_batch = tf.stack([self.__preprocess_image(data["x2"]) for data in batch])
            return x1_batch, x2_batch
        except Exception as e:
            print(f"Error in FaceVerificationSet.__getitem__ at index {idx}: {e}")
            return (tf.zeros((self.batch_size, *self.image_size, 3), dtype=tf.float32),
                    tf.zeros((self.batch_size, *self.image_size, 3), dtype=tf.float32))

# Convert FaceVerificationSet to tf.data.Dataset
def face_verification_to_dataset(face_verification):
    def generator():
        for idx in range(len(face_verification)):
            yield face_verification[idx]
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32)
        )
    )
    dataset = dataset.cache(filename=f'verification_cache_{random.randint(0, 1000000)}')
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    dataset = dataset.take(len(face_verification))
    return dataset

# Create and optimize datasets
start_time = time.time()
initial_embedding = mobilenetv2()
triplets_dataset = TripletFace("data/classification_data/train_data", batch_size=4, max_persons=20, embedding_model=initial_embedding, max_triplets=1000)  # Reduced for stability
val_triplets_dataset = TripletFace("data/classification_data/val_data", batch_size=4, max_persons=20, embedding_model=initial_embedding, max_triplets=100)  # Reduced for stability
triplets_dataset = triplet_face_to_dataset(triplets_dataset)
val_triplets_dataset = triplet_face_to_dataset(val_triplets_dataset)
print(f"Dataset initialization time: {time.time() - start_time:.2f} seconds")

cls_train_dataset = keras.utils.image_dataset_from_directory(
    "data/classification_data/train_data",
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    batch_size=16
)
cls_val_dataset = keras.utils.image_dataset_from_directory(
    "data/classification_data/val_data",
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    batch_size=16
)
cls_train_dataset = cls_train_dataset.map(lambda x, y: (keras.applications.mobilenet_v2.preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
cls_val_dataset = cls_val_dataset.map(lambda x, y: (keras.applications.mobilenet_v2.preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
cls_train_dataset = cls_train_dataset.cache(filename=f'cls_train_cache_{random.randint(0, 1000000)}').prefetch(tf.data.AUTOTUNE)
cls_val_dataset = cls_val_dataset.cache(filename=f'cls_val_cache_{random.randint(0, 1000000)}').prefetch(tf.data.AUTOTUNE)

# Train Siamese model with Euclidean distance
print("Training Siamese model with Euclidean distance...")
embedding_euclidean = mobilenetv2()
siamese_euclidean = SiameseModel(embedding_euclidean, loss="euclidean", margin=1.0)
siamese_euclidean.compile(
    optimizer=mixed_precision.LossScaleOptimizer(keras.optimizers.Adam(0.0001))
)

# Get dataset cardinality
train_cardinality = triplets_dataset.cardinality().numpy()
val_cardinality = val_triplets_dataset.cardinality().numpy()
print(f"Train dataset cardinality: {train_cardinality}")
print(f"Validation dataset cardinality: {val_cardinality}")

start_time = time.time()
siamese_euclidean_history = siamese_euclidean.fit(
    triplets_dataset,
    validation_data=val_triplets_dataset,
    epochs=5,
    steps_per_epoch=train_cardinality if train_cardinality > 0 else None,
    validation_steps=val_cardinality if val_cardinality > 0 else None,
    callbacks=[
        keras.callbacks.ModelCheckpoint("model_siamese_euclidean.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True)
    ]
)
print(f"Siamese Euclidean training time: {time.time() - start_time:.2f} seconds")

# Train Siamese model with Cosine distance
print("Training Siamese model with Cosine distance...")
embedding_cosine = mobilenetv2()
siamese_cosine = SiameseModel(embedding_cosine, loss="cosine", margin=1.0)
siamese_cosine.compile(
    optimizer=mixed_precision.LossScaleOptimizer(keras.optimizers.Adam(0.0001))
)
start_time = time.time()
siamese_cosine_history = siamese_cosine.fit(
    triplets_dataset,
    validation_data=val_triplets_dataset,
    epochs=5,
    steps_per_epoch=train_cardinality if train_cardinality > 0 else None,
    validation_steps=val_cardinality if val_cardinality > 0 else None,
    callbacks=[
        keras.callbacks.ModelCheckpoint("model_siamese_cosine.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True)
    ]
)
print(f"Siamese Cosine training time: {time.time() - start_time:.2f} seconds")

# Train classification model
print("Training classification model...")
embedding_classification = mobilenetv2()
classification = classification_head(embedding_classification)
classification.compile(
    optimizer=mixed_precision.LossScaleOptimizer(keras.optimizers.Adam(0.0001)),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
start_time = time.time()
classification.fit(
    cls_train_dataset,
    validation_data=cls_val_dataset,
    epochs=10,
    callbacks=[
        keras.callbacks.ModelCheckpoint("model_classification.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True)
    ]
)
print(f"Classification training time: {time.time() - start_time:.2f} seconds")

# Train anti-spoofing model (placeholder dataset)
print("Training anti-spoofing model...")
anti_spoof_dataset = cls_train_dataset.map(lambda x, y: (x, tf.zeros(tf.shape(x)[0], dtype=tf.float32)))
anti_spoof_val_dataset = cls_val_dataset.map(lambda x, y: (x, tf.zeros(tf.shape(x)[0], dtype=tf.float32)))
anti_spoof_model = anti_spoofing_model()
anti_spoof_model.compile(
    optimizer=mixed_precision.LossScaleOptimizer(keras.optimizers.Adam(0.001)),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
start_time = time.time()
anti_spoof_model.fit(
    anti_spoof_dataset,
    validation_data=anti_spoof_val_dataset,
    epochs=5,
    callbacks=[
        keras.callbacks.ModelCheckpoint("model_anti_spoof.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True)
    ]
)
print(f"Anti-spoofing training time: {time.time() - start_time:.2f} seconds")

# Evaluation
data_verification = FaceVerificationSet('data', 'data/verification_pairs_val.txt')
data_verification_dataset = face_verification_to_dataset(data_verification)

embedding_euclidean = mobilenetv2()
siamese_euclidean = SiameseModel(embedding_euclidean)
siamese_euclidean.load_weights("model_siamese_euclidean.weights.h5")
embedding_euclidean.save("embedding_euclidean.keras")
embedding_cosine = mobilenetv2()
siamese_cosine = SiameseModel(embedding_cosine, loss="cosine", margin=1.0)
siamese_cosine.load_weights("model_siamese_cosine.weights.h5")
embedding_cosine.save("embedding_cosine.keras")
embedding_classification = mobilenetv2()
classification = classification_head(embedding_classification)
classification.load_weights("model_classification.weights.h5")
embedding_classification.save("embedding_classification.keras")

siamese_euclidean_distance = distance_head(embedding_euclidean)
siamese_cosine_distance = distance_head(embedding_cosine, loss="cosine")
classification_distance = distance_head(embedding_classification)

# Anti-spoofing check before prediction
def verify_with_anti_spoof(image1, image2, distance_model, anti_spoof_model, threshold=0.5):
    spoof1 = anti_spoof_model.predict(tf.expand_dims(image1, axis=0))[0][0]
    spoof2 = anti_spoof_model.predict(tf.expand_dims(image2, axis=0))[0][0]
    if spoof1 > threshold or spoof2 > threshold:
        return float('inf')
    return distance_model.predict([tf.expand_dims(image1, axis=0), tf.expand_dims(image2, axis=0)])[0]

siamese_euclidean_preds = []
siamese_cosine_preds = []
classification_preds = []
for batch in data_verification_dataset:
    x1, x2 = batch
    for i in range(x1.shape[0]):
        pred_euclidean = verify_with_anti_spoof(x1[i], x2[i], siamese_euclidean_distance, anti_spoof_model)
        pred_cosine = verify_with_anti_spoof(x1[i], x2[i], siamese_cosine_distance, anti_spoof_model)
        pred_classification = verify_with_anti_spoof(x1[i], x2[i], classification_distance, anti_spoof_model)
        siamese_euclidean_preds.append(pred_euclidean)
        siamese_cosine_preds.append(pred_cosine)
        classification_preds.append(pred_classification)

siamese_euclidean_fpr, siamese_euclidean_tpr, siamese_euclidean_thresholds = roc_curve(data_verification.get_y(), siamese_euclidean_preds)
siamese_cosine_fpr, siamese_cosine_tpr, siamese_cosine_thresholds = roc_curve(data_verification.get_y(), siamese_cosine_preds)
classification_fpr, classification_tpr, classification_thresholds = roc_curve(data_verification.get_y(), classification_preds)

siamese_euclidean_auc = roc_auc_score(data_verification.get_y(), siamese_euclidean_preds)
siamese_cosine_auc = roc_auc_score(data_verification.get_y(), siamese_cosine_preds)
classification_auc = roc_auc_score(data_verification.get_y(), classification_preds)

plt.plot(siamese_euclidean_fpr, siamese_euclidean_tpr, label=f"Siamese with Euclidean Distance Loss (AUC: {siamese_euclidean_auc:.2f})")
plt.plot(siamese_cosine_fpr, siamese_cosine_tpr, label=f"Siamese with Cosine Distance Loss (AUC: {siamese_cosine_auc:.2f})")
plt.plot(classification_fpr, classification_tpr, label=f"Classification (AUC: {classification_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

siamese_euclidean_best_threshold_idx = np.argmax(siamese_euclidean_tpr - siamese_euclidean_fpr)
print(f"Siamese with Euclidean Distance Loss best threshold: {siamese_euclidean_thresholds[siamese_euclidean_best_threshold_idx]}")
print(f"Siamese with Euclidean Distance Loss best fpr: {siamese_euclidean_fpr[siamese_euclidean_best_threshold_idx]}")
print(f"Siamese with Euclidean Distance Loss best tpr: {siamese_euclidean_tpr[siamese_euclidean_best_threshold_idx]}")

siamese_cosine_best_threshold_idx = np.argmax(siamese_cosine_tpr - siamese_cosine_fpr)
print(f"Siamese with Cosine Distance Loss best threshold: {siamese_cosine_thresholds[siamese_cosine_best_threshold_idx]}")
print(f"Siamese with Cosine Distance Loss best fpr: {siamese_cosine_fpr[siamese_cosine_best_threshold_idx]}")
print(f"Siamese with Cosine Distance Loss best tpr: {siamese_cosine_tpr[siamese_cosine_best_threshold_idx]}")

classification_best_threshold_idx = np.argmax(classification_tpr - classification_fpr)
print(f"Classification best threshold: {classification_thresholds[classification_best_threshold_idx]}")
print(f"Classification best fpr: {classification_fpr[classification_best_threshold_idx]}")
print(f"Classification best tpr: {classification_tpr[classification_best_threshold_idx]}")